# C06 — Detecção de Antinomias (Conflitos Candidatos entre Normas)

**Disciplina:** Sistemas Cognitivos com Large Language Models (INFNET, 26E2_3)
**Autor:** Anderson Felipe Paixão Corrêa
**Projeto:** Letra da Lei — RAG hierarquia- e vigência-aware sobre o microssistema
penal federal brasileiro

Esta notebook cobre a **especificação de design §5/§6.4** — o detector de antinomias, a
contribuição mais original do projeto. Ela **empacota** módulos já implementados e
testados do pacote `direito_dados` (`direito_dados.conflicts`) — nenhuma lógica de geração
de candidatos, adjudicação ou avaliação é reimplementada aqui, apenas chamada e narrada.

O pipeline, em três etapas:

1. **Geração de candidatos** (`generate_candidates`) — recuperação por similaridade
   restringe o espaço O(n²) de pares de dispositivos a pares topicamente relacionados, que
   *poderiam* estar em conflito.
2. **Adjudicação** (`detect_conflicts`) — para cada candidato, um princípio de resolução
   LINDB é calculado deterministicamente quando possível (*lex superior*/*lex posterior*)
   e servido como dica ao modelo local, que julga se há conflito plausível e (quando a dica
   é indeterminada) refina para *lex specialis*.
3. **Avaliação** (`evaluate_antinomias`) — precisão/revocação/F1 contra um gold-set
   pequeno e **ilustrativo**, com a limitação de não ter sido verificado por especialista
   explicitamente declarada.

**Ponto central, repetido ao longo da notebook:** tudo que sai deste pipeline é um
**candidato para revisão humana**, nunca um veredito. Nenhuma aresta `conflict_candidate`
do grafo (`direito_dados.graph`) é gerada com `verification_state` diferente de
`CANDIDATE`.

## Setup: índice sobre o Código Penal

In [1]:
import sys
import time
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "direito_dados").exists():
    _repo_root = Path(__file__).resolve().parent if "__file__" in globals() else _repo_root
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from direito_dados.corpus import load_corpus, NORMS
from direito_dados.corpus.models import HierarchyLevel
from direito_dados.retrieval.chunks import chunk_corpus
from direito_dados.retrieval.embedder import E5Embedder
from direito_dados.retrieval.index import VectorIndex
from direito_dados.generation.llm import OllamaClient, ollama_available, ollama_has_model
from direito_dados.conflicts.candidates import generate_candidates
from direito_dados.conflicts.detect import detect_conflicts
from direito_dados.conflicts.principles import deterministic_principle, principle_hint, ResolutionPrinciple
from direito_dados.conflicts.evaluate import GoldAntinomy, evaluate_antinomias

MODEL = "llama3.1:8b"
assert ollama_available(), "Ollama precisa estar rodando em localhost:11434"
assert ollama_has_model(MODEL), f"Modelo {MODEL} precisa estar baixado (`ollama pull {MODEL}`)"
llm = OllamaClient(model=MODEL)

t0 = time.time()
corpus = load_corpus("data/raw", specs=[NORMS["CP"]])
chunks = chunk_corpus(corpus)
embedder = E5Embedder()
index = VectorIndex.build(chunks, embedder)
chunks_by_id = {c.id: c for c in chunks}
print(f"Índice construído em {time.time() - t0:.1f}s | {len(chunks)} dispositivos do CP indexados")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Índice construído em 12.2s | 431 dispositivos do CP indexados


## 1. Geração de candidatos

`generate_candidates` consulta, para cada dispositivo em vigor, seus `k` vizinhos
semânticos mais próximos (excluindo revogados) e mantém pares cuja similaridade máxima
ultrapassa `threshold`. Usamos os mesmos parâmetros do teste de integração do projeto
(`k=3`, `threshold=0.85`) — um limiar alto, deliberado: candidatos são caros (cada um vira
uma chamada ao LLM na etapa de adjudicação), então a geração prioriza precisão sobre
recall nesta fase, aceitando perder pares de conflito mais sutis (recall menor) em troca de
não inundar a adjudicação com pares irrelevantes.

In [2]:
t0 = time.time()
candidates = generate_candidates(chunks, index, embedder, k=3, threshold=0.85)
print(f"Candidatos gerados em {time.time() - t0:.1f}s: {len(candidates)} pares acima do limiar")
print("\nTop 15 por similaridade:")
for cp in candidates[:15]:
    print(f"  {cp.a:<16} × {cp.b:<16} sim={cp.similarity:.3f}")

Candidatos gerados em 8.6s: 910 pares acima do limiar

Top 15 por similaridade:
  CP:art210        × CP:art211        sim=0.908
  CP:art238        × CP:art239        sim=0.906
  CP:art326        × CP:art337-J      sim=0.903
  CP:art125        × CP:art126        sim=0.901
  CP:art296        × CP:art306        sim=0.900
  CP:art209        × CP:art210        sim=0.899
  CP:art197        × CP:art199        sim=0.899
  CP:art124        × CP:art126        sim=0.899
  CP:art276        × CP:art277        sim=0.898
  CP:art198        × CP:art199        sim=0.898
  CP:art213        × CP:art217-A      sim=0.898
  CP:art359-O      × CP:art359-Q      sim=0.897
  CP:art197        × CP:art198        sim=0.897
  CP:art289        × CP:art306        sim=0.894
  CP:art18         × CP:art19         sim=0.893


**Leitura:** com um corpus de 431 dispositivos vigentes do CP, a comparação exaustiva seria
~92 mil pares; a restrição por similaridade reduz isso a algumas centenas de candidatos
plausíveis — ainda demais para adjudicar todos com um LLM local em tempo de notebook. Por
isso a próxima etapa roda sobre um subconjunto limitado (os 12 pares de maior
similaridade), coerente com a orientação do projeto de manter as chamadas ao Ollama
limitadas nesta notebook.

## 2. Princípios LINDB determinísticos vs. adjudicação por LLM

Antes de adjudicar, vale explicitar a divisão de trabalho: *lex superior* (hierarquia) e
*lex posterior* (data) são **computáveis deterministicamente** a partir de metadados —
nenhum LLM é necessário nem deveria ser usado para essas duas. *Lex specialis*
(especificidade — qual norma trata do caso de forma mais específica) exige **ler e
comparar o conteúdo** dos dois dispositivos, então fica a cargo do modelo.

Como todos os 431 dispositivos aqui vêm do mesmo Código Penal (mesma hierarquia, mesma
data de origem), `principle_hint` sempre devolve "avaliar lex specialis" para os pares do
Código Penal — o caso determinístico só aparece comparando normas de hierarquias/datas
diferentes. Demonstramos os três ramos com exemplos sintéticos de metadados (sem chamar o
LLM, já que a lógica é 100% determinística):

In [3]:
exemplos = [
    ("CF vs. CP (hierarquias diferentes)", HierarchyLevel.CONSTITUICAO, HierarchyLevel.DECRETO_LEI, 1988, 1940),
    ("Duas leis ordinárias, anos diferentes", HierarchyLevel.LEI_ORDINARIA, HierarchyLevel.LEI_ORDINARIA, 1990, 2006),
    ("Dois artigos do mesmo CP (caso real desta notebook)", HierarchyLevel.DECRETO_LEI, HierarchyLevel.DECRETO_LEI, None, None),
]
for label, level_a, level_b, year_a, year_b in exemplos:
    principio = deterministic_principle(level_a, level_b, year_a, year_b)
    dica = principle_hint(level_a, level_b, year_a, year_b)
    print(f"{label}:\n  princípio determinístico = {principio.value}\n  dica ao LLM = {dica!r}\n")

CF vs. CP (hierarquias diferentes):
  princípio determinístico = lex_superior
  dica ao LLM = 'Hierarquia distinta: lex superior pode prevalecer (norma superior derroga inferior).'

Duas leis ordinárias, anos diferentes:
  princípio determinístico = lex_posterior
  dica ao LLM = 'Mesma hierarquia, datas distintas: lex posterior pode prevalecer (norma posterior derroga anterior).'

Dois artigos do mesmo CP (caso real desta notebook):
  princípio determinístico = undetermined
  dica ao LLM = 'Mesma hierarquia e data: avaliar lex specialis (a norma mais específica pode prevalecer).'



## 3. Adjudicação: candidatos → conflitos candidatos

`detect_conflicts` adjudica cada par com o LLM local: constrói o hint de princípio,
apresenta os dois textos, e mantém os pares em que o modelo julgou `conflict=true` com
`confidence >= min_confidence`. O `principle` final vem do modelo quando ele se posiciona
(tipicamente `lex_specialis`, aqui); cai para o princípio determinístico só quando o modelo
não se posiciona.

In [4]:
top12 = candidates[:12]
t0 = time.time()
conflicts = detect_conflicts(top12, chunks_by_id, corpus, llm, min_confidence=0.5)
print(f"Adjudicados {len(top12)} pares em {time.time() - t0:.1f}s -> {len(conflicts)} confirmados como conflito candidato\n")
for c in conflicts:
    print(f"{c.a:<16} × {c.b:<16} | princípio={c.principle:<14} confiança={c.confidence:.2f}")
    if c.rationale:
        print(f"    razão: {c.rationale}")

Adjudicados 12 pares em 30.5s -> 12 confirmados como conflito candidato

CP:art210        × CP:art211        | princípio=lex_specialis  confiança=1.00
CP:art238        × CP:art239        | princípio=lex_specialis  confiança=1.00
CP:art326        × CP:art337-J      | princípio=lex_specialis  confiança=0.90
CP:art125        × CP:art126        | princípio=lex_specialis  confiança=0.80
CP:art296        × CP:art306        | princípio=lex_specialis  confiança=0.80
CP:art209        × CP:art210        | princípio=lex_specialis  confiança=0.80
    razão: Ambas as normas tratam de violação a sepulturas, mas o art. 210 (Dispositivo B) aborda especificamente 'violar ou profanar sepultura ou urna funerária', sendo considerado mais específico e, portanto, prevalecente.
CP:art197        × CP:art199        | princípio=lex_specialis  confiança=1.00
CP:art124        × CP:art126        | princípio=lex_specialis  confiança=0.90
CP:art276        × CP:art277        | princípio=lex_specialis  confiança=0.90


**Leitura:** dos 12 pares adjudicados, **todos os 12** foram confirmados como conflito candidato
nesta execução — esperado, já que a etapa de geração (seção 1) já filtrou por altíssima similaridade
temática, então quase todos os pares realmente tratam de matéria sobreposta. **Importante:**
"confirmado" aqui significa apenas que o LLM local julgou plausível uma antinomia digna de
revisão humana — não que exista de fato uma contradição normativa juridicamente
estabelecida. Nenhuma dessas linhas vira aresta `conflict_candidate` no grafo com
`verification_state` diferente de `CANDIDATE` (`direito_dados.conflicts.detect.add_conflict_edges`),
e a taxa de confirmação alta é, em si, um sinal de que o **limiar de similaridade da geração
de candidatos, não a adjudicação, é quem está fazendo a triagem mais forte** — outro
lembrete de que os números desta seção dependem fortemente de `k`/`threshold` (seção 1).

## 4. Avaliação: gold-set ilustrativo

Para medir precisão/revocação, construímos um gold-set de **3 pares**, cada um justificado
por conhecimento jurídico geral (não por consulta a um especialista em direito penal — essa
é a limitação central desta seção, declarada explicitamente):

- **`CP:art213` × `CP:art217-A`** (estupro / estupro de vulnerável): fricção doutrinária
  real e conhecida — quando a vítima é vulnerável *e* há violência ou grave ameaça, a
  aplicação isolada de um ou outro tipo penal, ou o concurso entre eles, é objeto de debate
  na doutrina e na jurisprudência penal brasileira. Um caso canônico de possível
  *lex specialis* (art. 217-A é mais específico quanto ao sujeito passivo).
- **`CP:art197` × `CP:art198`** (atentado contra a liberdade de trabalho / constranger a
  celebrar contrato de trabalho): relação clássica de norma geral vs. norma especial — o
  art. 197 tipifica o constrangimento à liberdade de trabalho de forma genérica, e o art.
  198 tipifica uma manifestação específica dessa mesma conduta (forçar a celebração de um
  contrato de trabalho específico).
- **`CP:art124` × `CP:art126`** (autoaborto/aborto consentido pela gestante / aborto
  provocado por terceiro com consentimento da gestante): a família de crimes de aborto do
  CP distribui a mesma conduta nuclear (interrupção da gravidez) por diferentes graus de
  participação e consentimento — um terreno clássico de sobreposição normativa discutido em
  doutrina de direito penal.

**Isto não é um gold-set validado por um especialista em direito penal** — é uma seleção
justificada, mas subjetiva, feita pelo autor deste projeto a partir da leitura dos próprios
candidatos gerados. Um gold-set real (~15 pares, conforme a especificação de design §5.1)
exigiria revisão por um profissional da área, fora do escopo deste projeto acadêmico.

In [5]:
gold = [
    GoldAntinomy("CP:art213", "CP:art217-A"),
    GoldAntinomy("CP:art197", "CP:art198"),
    GoldAntinomy("CP:art124", "CP:art126"),
]

metrics = evaluate_antinomias(conflicts, gold)
print(f"precisão:  {metrics['precision']:.3f}  (tp={metrics['tp']}, fp={metrics['fp']})")
print(f"revocação: {metrics['recall']:.3f}  (tp={metrics['tp']}, fn={metrics['fn']})")
print(f"F1:        {metrics['f1']:.3f}")

precisão:  0.167  (tp=2, fp=10)
revocação: 0.667  (tp=2, fn=1)
F1:        0.267


**Leitura:** a precisão baixa observada aqui **não** significa que os demais pares
detectados (que não batem com o gold-set de 3 itens) sejam falsos positivos no sentido
jurídico — a maioria são candidatos igualmente plausíveis que simplesmente não foram
incluídos neste gold-set minúsculo e ilustrativo. Com apenas 3 pares no gold-set contra os 12
conflitos candidatos confirmados nesta execução, a precisão numérica é estruturalmente baixa por
construção, não porque o detector erre sistematicamente. O par `CP:art197` × `CP:art198`
ilustra o outro lado do problema: ele está no gold-set mas pode não aparecer nos
`conflicts` se não estiver entre os 12 candidatos de maior similaridade adjudicados na
seção 3 — nesse caso, o par é uma revocação perdida não por o modelo tê-lo julgado
"sem conflito", mas por a **etapa de geração de candidatos** nunca tê-lo submetido à
adjudicação. Essa é exatamente a mesma lição da seção 3: o funil de similaridade/limiar
determina o teto de recall antes mesmo do LLM entrar em cena.

Este resultado é ilustrativo do método de avaliação, não uma medida confiável de qualidade
do detector — o gold-set real de ~15 pares (design spec §5.1), verificado por um
especialista em direito penal, é trabalho futuro declarado, não uma lacuna escondida.

## Conclusão

Esta notebook demonstrou o detector de antinomias completo sobre o Código Penal: geração
de candidatos por similaridade semântica (§1), a separação de responsabilidades entre
princípios LINDB determinísticos (*lex superior*/*lex posterior*, computados de metadados,
sem LLM) e a adjudicação por LLM da *lex specialis* — que exige leitura e comparação de
conteúdo (§2–3) —, e uma avaliação precisão/revocação/F1 contra um gold-set pequeno e
declaradamente ilustrativo (§4). O ponto mais importante, repetido deliberadamente: **cada
conflito aqui é um candidato para revisão humana, nunca um veredito** — a mesma postura de
transparência de candidato que rege `hallucinated_citations` em `c05_rag_pipeline.ipynb`.
A qualidade de ponta a ponta do detector é limitada, antes de tudo, pelo limiar de geração
de candidatos (§1/§3) e pela cobertura do gold-set (§4) — ambas são superfícies de ajuste
explícitas para trabalho futuro, não falhas escondidas do pipeline.